In [ ]:
# 1. 필수 라이브러리 및 LLaMA-Factory 설치
# (최신 버전 충돌 방지를 위해 transformers 버전을 4.57.1로 고정합니다)
!git clone https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory

# 의존성 지옥 방지용 강제 버전 고정 설치
!pip install "transformers==4.57.1" "huggingface_hub<1.0.0"
!pip install -e .[metrics]
!pip install -q qwen_vl_utils bitsandbytes scipy
!pip install flash-attn --no-build-isolation

# 필수 라이브러리 설치
!pip install faker tqdm

print("✅ 환경 설정 완료! [런타임] -> [세션 다시 시작]을 누르고 2단계로 넘어가세요.")

Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 26100, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 26100 (delta 10), reused 3 (delta 3), pack-reused 26060 (from 2)
Receiving objects: 100% (26100/26100), 12.59 MiB | 17.26 MiB/s, done.
Resolving deltas: 100% (18792/18792), done.
/content/LLaMA-Factory
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 133.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6
Obtaining file:///content/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.

In [7]:
import os
import json
import random
import requests
import shutil
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from faker import Faker
from tqdm import tqdm

# --- 설정 ---
BASE_DIR = "/content"
DATA_DIR = os.path.join(BASE_DIR, "data_vlm")
TEMPLATE_PATH = Path("/content/waybill_template.jpg")
FONTS_DIR = Path("/content/fonts")
GENERATE_COUNT = 5000  # 3만장은 너무 많으니 5000장으로 테스트 (충분함)

# --- 0. 환경 초기화 ---
if os.path.exists(DATA_DIR): shutil.rmtree(DATA_DIR)
os.makedirs(os.path.join(DATA_DIR, "images"), exist_ok=True)
if not FONTS_DIR.exists(): FONTS_DIR.mkdir()

# --- 1. 폰트 다운로드 ---
FONT_URLS = {
    "NanumGothicBold": "https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Bold.ttf",
    "DoHyeon": "https://github.com/google/fonts/raw/main/ofl/dohyeon/DoHyeon-Regular.ttf",
}
for name, url in FONT_URLS.items():
    if not (FONTS_DIR / f"{name}.ttf").exists():
        try:
            r = requests.get(url)
            with open(FONTS_DIR / f"{name}.ttf", 'wb') as f: f.write(r.content)
        except: pass

def get_font():
    fonts = list(FONTS_DIR.glob("*.ttf"))
    return str(random.choice(fonts)) if fonts else "arial.ttf"

# --- 2. 데이터 생성기 ---
fake = Faker('ko_KR')
class VLMDataGenerator:
    def __init__(self):
        if not TEMPLATE_PATH.exists():
            raise FileNotFoundError("❌ waybill_template.jpg 파일이 없습니다! 업로드해주세요.")
        self.bg = Image.open(TEMPLATE_PATH).convert("RGB")
        self.positions = {
            'tracking_number': (50, 50, 400, 100),
            'region_code': (600, 50, 750, 100),
            'recipient_name': (150, 150, 500, 200),
            'recipient_address': (50, 200, 750, 300),
            'sender_name': (150, 320, 500, 370),
            'sender_address': (50, 370, 750, 470)
        }

    def render(self):
        img = self.bg.copy()
        draw = ImageDraw.Draw(img)
        data = {
            "운송장번호": ''.join([str(random.randint(0, 9)) for _ in range(12)]),
            "분류코드": f"{fake.city()[:2]}{random.randint(1, 9)}",
            "받는분": fake.name(),
            "받는분주소": fake.address(),
            "보내는분": fake.name(),
            "보내는분주소": fake.address()
        }
        font_path = get_font()
        for key, text in data.items():
            pos_map = {
                "운송장번호": 'tracking_number', "분류코드": 'region_code',
                "받는분": 'recipient_name', "받는분주소": 'recipient_address',
                "보내는분": 'sender_name', "보내는분주소": 'sender_address'
            }
            if key not in pos_map: continue
            x1, y1, x2, y2 = self.positions[pos_map[key]]
            size = 40
            font = ImageFont.truetype(font_path, size)
            while font.getlength(text) > (x2 - x1) and size > 15:
                size -= 2
                font = ImageFont.truetype(font_path, size)
            draw.text((x1, y1), text, font=font, fill=(30, 30, 30))
        return img, data

# --- 3. 생성 실행 ---
generator = VLMDataGenerator()
llama_data = []

print(f"🚀 데이터 {GENERATE_COUNT}장 재생성 중...")
for i in tqdm(range(GENERATE_COUNT)):
    img, json_data = generator.render()
    filename = f"{i:05d}.jpg"
    img_path = os.path.join(DATA_DIR, "images", filename)
    img.save(img_path)

    entry = {
        "messages": [
            {"role": "user", "content": "<image>이 운송장 이미지에서 정보를 추출하여 JSON 형식으로 출력해줘."},
            {"role": "assistant", "content": json.dumps(json_data, ensure_ascii=False)}
        ],
        "images": [img_path]
    }
    llama_data.append(entry)

train_file = os.path.join(DATA_DIR, "train_data.json")
with open(train_file, "w", encoding='utf-8') as f:
    json.dump(llama_data, f, indent=2, ensure_ascii=False)

# --- 4. LLaMA-Factory 등록 ---
info = {
  "waybill_vlm": {
    "file_name": train_file,
    "formatting": "sharegpt",
    "columns": {"messages": "messages", "images": "images"},
    "tags": {"role_tag": False, "image_tag": False, "zero_shot": True}
  }
}
info_path = "/content/LLaMA-Factory/data/dataset_info.json"
with open(info_path, "r") as f: original_info = json.load(f)
original_info.update(info)
with open(info_path, "w") as f: json.dump(original_info, f, indent=2)

print("✅ 데이터 생성완료 및 등록 완료! 이제 학습하세요.")

🚀 데이터 5000장 재생성 중...


100%|██████████| 5000/5000 [01:22<00:00, 60.97it/s]


✅ 데이터 복구 및 등록 완료! 이제 학습하세요.


In [9]:
# 학습 명령어 실행 (Epoch 3, 약 10~15분 소요)
!llamafactory-cli train \
    --stage sft \
    --do_train \
    --model_name_or_path Qwen/Qwen2-VL-2B-Instruct \
    --dataset waybill_vlm \
    --template qwen2_vl \
    --finetuning_type lora \
    --lora_target all \
    --output_dir /content/qwen2_vl_finetuned \
    --per_device_train_batch_size 16 \
    --gradient_accumulation_steps 4 \
    --learning_rate 1e-4 \
    --num_train_epochs 3.0 \
    --max_grad_norm 1.0 \
    --lr_scheduler_type cosine \
    --warmup_ratio 0.1 \
    --fp16 True \
    --plot_loss True \
    --logging_steps 10 \
    --save_steps 500 \
    --save_total_limit 2 \
    --overwrite_output_dir

2026-01-30 08:45:11.172167: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-30 08:45:11.177556: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-30 08:45:11.193106: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769762711.221587    2388 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769762711.230100    2388 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769762711.250139    2388 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch
from PIL import Image
import json

# 1. 학습된 모델 로드 (LoRA 어댑터 결합)
print("🤖 학습된 모델을 불러오는 중입니다...")
model_path = "/content/qwen2_vl_finetuned"
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct", torch_dtype=torch.float16, device_map="auto"
)
model.load_adapter(model_path)
model.merge_and_unload() # 추론 속도 최적화

processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")

# 2. 테스트할 이미지 선택 (방금 만든 데이터 중 0번째 파일)
test_img_path = "/content/data_vlm/images/00000.jpg"
image = Image.open(test_img_path)
print("📸 테스트 이미지:")
display(image.resize((400, 250)))

# 3. 질문 던지기
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": test_img_path},
            {"type": "text", "text": "이 운송장 이미지에서 정보를 추출하여 JSON 형식으로 출력해줘."},
        ],
    }
]

# 4. 추론 실행
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
).to("cuda")

print("⏳ AI가 분석 중...")
generated_ids = model.generate(**inputs, max_new_tokens=512)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)

print("\n🎉 [최종 결과] :")
print(output_text[0])